# SQL 기초 — 정렬·가공과 집계  (SQLite 노트북)
원본 파일: `w5_SQL_기초_정렬_가공과_집계.sql` · SQL 셀 40개

## 사용법
1. **위에서부터 순서대로** 실행하세요. 앞쪽 셀이 테이블(`customers`, `orders` 등)을 만듭니다. 데이터 생성 셀은 `DROP TABLE IF EXISTS`로 시작해서 다시 실행해도 안전합니다.
2. 연습 문제 셀은 주석 아래에 SQL을 쓰고 실행합니다. 원본 파일에 이미 풀이가 들어 있던 셀은 그대로 가져왔습니다.
3. 준비물: Python 3, `pandas` (`pip install pandas`), VS Code의 Python·Jupyter 확장. `sqlite3`는 파이썬에 내장되어 있어서 설치가 필요 없습니다.

## 원본(BigQuery/DuckDB)에서 자동 변환한 내용
- 테이블 경로 `` `aiffel-515805.modumall.x` ``, `project_name.dataset_name.x` → `x`
- `CREATE OR REPLACE TABLE` → `DROP TABLE IF EXISTS` + `CREATE TABLE`, 열 타입 `STRING`/`INT64`/`DECIMAL`/`DATE` → `TEXT`/`INTEGER`/`REAL`/`TEXT`
- 날짜는 SQLite에 별도 타입이 없어서 `'YYYY-MM-DD'` 텍스트로 저장합니다.

### 셀별 추가 변환
- **C29**: 원본의 끝 마침표(.) 오타 제거
- **C33**: EXTRACT(… FROM 날짜) → CAST(strftime(…) AS INTEGER)
- **C36**: FROM 앞의 마지막 쉼표 제거
- **C38**: DATE_TRUNC → date(열, 'start of month')
- **C39**: DATE_TRUNC → date(열, 'start of month')

> 검증: 원본 SQL을 DuckDB에서, 변환본을 SQLite에서 각각 실행해 셀별 결과를 비교했습니다. 값이 다른 셀은 없었습니다. (정렬 기준이 없는 쿼리는 행 순서가, 열 이름을 직접 안 붙인 `count(*)` 같은 열은 자동 이름이 엔진마다 다를 수 있습니다.)

In [1]:
import re, sqlite3
import pandas as pd
try:
    from IPython.display import display
except ImportError:          # 일반 파이썬에서 실행할 때
    display = print

con = sqlite3.connect(":memory:")   # 메모리 DB: 노트북(커널)을 끄면 사라집니다

def sql(text):
    """SQL을 실행합니다. SELECT류는 표로 보여주고, CREATE/INSERT 등은 실행만 합니다."""
    stmts, buf = [], ""
    for line in text.splitlines(keepends=True):
        buf += line
        if sqlite3.complete_statement(buf):      # ; 로 끝나는 문장 단위로 분리
            stmts.append(buf.strip()); buf = ""
    if buf.strip():
        stmts.append(buf.strip())
    ran = shown = 0
    for st in stmts:
        body = re.sub(r"--[^\n]*|/\*.*?\*/", "", st, flags=re.S).strip().rstrip(";").strip()
        if not body:
            continue                              # 주석만 있는 조각은 건너뜀
        ran += 1
        if body.split(None, 1)[0].upper() in ("SELECT", "WITH", "PRAGMA", "VALUES"):
            display(pd.read_sql_query(st, con)); shown += 1
        else:
            con.execute(st)
    con.commit()
    if ran == 0:
        print("(아직 작성한 SQL이 없습니다. 주석 아래에 쿼리를 작성한 뒤 다시 실행하세요.)")
    elif shown == 0:
        print("실행 완료")

print("SQLite 버전:", sqlite3.sqlite_version)

SQLite 버전: 3.53.4


## 1. 두 환경의 문법 차이 — 한눈에 보기

> **[C1] 건너뜀** — 원본의 DuckDB/BigQuery용 이름 공간 준비 셀입니다. SQLite는 필요 없습니다.

In [2]:
sql(r"""
-- [C2]
DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
  customer_id TEXT,
  name TEXT,
  country TEXT,
  signup_date TEXT,
  grade TEXT
);

INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze')
""")

실행 완료


In [3]:
sql(r"""
-- [C3]
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
  order_id TEXT,
  customer_id TEXT,
  order_date TEXT,
  status TEXT,
  amount REAL
);

INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000)
""")

실행 완료


In [4]:
sql(r"""
-- [C4]
DROP TABLE IF EXISTS products;
CREATE TABLE products (
  product_id TEXT,
  product_name TEXT,
  category TEXT,
  price REAL
);

INSERT INTO products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000)
""")

실행 완료


### 2. ▶️ **코드 실행하기 · 코드 셀 5 [C5]**

In [5]:
sql(r"""
-- [C5]
SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  amount IS NOT NULL
ORDER BY
  amount DESC
LIMIT 5
""")

,order_id,status,amount
0,O0009,Shipped,410000.0
1,O0027,Shipped,405000.0
2,O0018,Paid,320000.0
3,O0024,Paid,268000.0
4,O0004,Paid,230000.0


### 3. ▶️ **코드 실행하기 · 코드 셀 6 [C6]**

In [6]:
sql(r"""
-- [C6]
SELECT
  order_id,
  order_date,
  amount
FROM orders
ORDER BY
  order_date NULLS LAST
LIMIT 5
""")

,order_id,order_date,amount
0,O0001,2023-09-02,125000.0
1,O0002,2023-09-05,89000.0
2,O0003,2023-09-11,45000.0
3,O0004,2023-09-15,230000.0
4,O0005,2023-09-20,NaN


## 4. ⌨️ 백문이 불여일타 (1)

In [7]:
sql(r"""
-- [C7] ⌨️ 백문이 불여일타 (1) — customers 가입일 최근 순 정렬

SELECT
  name,
  signup_date
FROM customers
ORDER BY
  signup_date DESC
""")

,name,signup_date
0,안수아,2023-08-28
1,황지안,2023-08-10
2,권준서,2023-07-25
3,신아린,2023-07-07
4,오시우,2023-06-19
5,한예준,2023-06-02
6,임하은,2023-05-22
7,윤지호,2023-05-09
8,조은우,2023-04-18
9,강서윤,2023-04-01


## 5. ⌨️ 백문이 불여일타 (2)

In [8]:
sql(r"""
-- [C8] ⌨️ 백문이 불여일타 (2) — 가장 비싼 상품 TOP 3

SELECT
  product_name,
  price
FROM products
ORDER BY
  price DESC
LIMIT 3
""")

,product_name,price
0,워커 부츠,175000.0
1,첼시 부츠,145000.0
2,트레일 러너,119000.0


### 6. ▶️ **코드 실행하기 · 코드 셀 9 [C9]**

In [9]:
sql(r"""
-- [C9]
SELECT
  name AS 고객명,
  country AS 국가,
  grade AS 등급
FROM customers
LIMIT 5
""")

,고객명,국가,등급
0,김민준,Korea,Gold
1,이서연,Korea,Silver
2,박도윤,Japan,Bronze
3,최지우,USA,Gold
4,정하준,Korea,Silver


### 7. ▶️ **코드 실행하기 · 코드 셀 10 [C10]**

In [10]:
sql(r"""
-- [C10]
SELECT
  product_name,
  category,
  price
FROM products
WHERE
  product_name LIKE '%스니커즈%'
""")

,product_name,category,price
0,클래식 스니커즈,Sneakers,65000.0
1,캔버스 스니커즈,Sneakers,49000.0


### 8. ▶️ **코드 실행하기 · 코드 셀 11 [C11]**

In [11]:
sql(r"""
-- [C11]
SELECT DISTINCT
  status
FROM orders
WHERE
  status LIKE 'P%'
ORDER BY
  status
""")

,status
0,Paid
1,Placed


### 9. ▶️ **코드 실행하기 · 코드 셀 12 [C12]**

In [12]:
sql(r"""
-- [C12]
SELECT DISTINCT
  status
FROM orders
WHERE
  status LIKE 'P___'
ORDER BY
  status
""")

,status
0,Paid


## 10. ⌨️ 백문이 불여일타 (3)

In [13]:
sql(r"""
-- [C13] ⌨️ 백문이 불여일타 (3) — 고객 등급 종류 중복 없이 조회

SELECT DISTINCT
  grade
FROM customers
""")

,grade
0,Gold
1,Silver
2,Bronze
3,NaN


## 11. ⌨️ 백문이 불여일타 (4)

In [14]:
sql(r"""
-- [C14] ⌨️ 백문이 불여일타 (4) — 부츠 상품 한글 별칭 조회

SELECT
  product_name AS 상품명,
  price AS 가격
FROM products
WHERE
  product_name LIKE '%부츠%'
  
""")

,상품명,가격
0,첼시 부츠,145000.0
1,워커 부츠,175000.0


### 12. ▶️ **코드 실행하기 · 코드 셀 15 [C15]**

In [15]:
sql(r"""
-- [C15]
SELECT
  order_id,
  amount,
  CASE WHEN amount >= 200000 THEN '고액' WHEN amount >= 100000 THEN '중액' ELSE '소액' END AS amount_class
FROM orders
WHERE
  amount IS NOT NULL
ORDER BY
  amount DESC
""")

,order_id,amount,amount_class
0,O0009,410000.0,고액
1,O0027,405000.0,고액
2,O0018,320000.0,고액
3,O0024,268000.0,고액
4,O0004,230000.0,고액
5,O0020,215000.0,고액
6,O0030,187000.0,중액
7,O0016,175000.0,중액
8,O0007,158000.0,중액
9,O0013,142000.0,중액


## 13. ⌨️ 백문이 불여일타 (5)

In [16]:
sql(r"""
-- [C16] ⌨️ 백문이 불여일타 (5) — 주문 상태 실패/정상 분류

SELECT
  order_id,
  status,
  amount,
  CASE
    WHEN status IN ('Cancelled', 'Returned') THEN '실패'
    ELSE '정상'
  END AS result
FROM orders
""")

,order_id,status,amount,result
0,O0001,Paid,125000.0,정상
1,O0002,Shipped,89000.0,정상
2,O0003,Returned,45000.0,실패
3,O0004,Paid,230000.0,정상
4,O0005,Cancelled,NaN,실패
5,O0006,Shipped,67000.0,정상
6,O0007,Paid,158000.0,정상
7,O0008,Placed,32000.0,정상
8,O0009,Shipped,410000.0,정상
9,O0010,Paid,99000.0,정상


## 14. ⌨️ 백문이 불여일타 (6)

In [17]:
sql(r"""
-- [C17] ⌨️ 백문이 불여일타 (6) — 가격 3단 분류 tier 열 만들기

select 
  product_name, 
  price, 
  case 
    when price >= 100000 then '프리미엄'
    when price >= 30000 then '스탠다드'
    else '베이직'
  end as tier
from products
order by price desc
""")

,product_name,price,tier
0,워커 부츠,175000.0,프리미엄
1,첼시 부츠,145000.0,프리미엄
2,트레일 러너,119000.0,프리미엄
3,에어러너,89000.0,스탠다드
4,클래식 스니커즈,65000.0,스탠다드
5,캔버스 스니커즈,49000.0,스탠다드
6,여름 샌들,38000.0,스탠다드
7,슬리퍼 샌들,25000.0,베이직
8,양말 세트,12000.0,베이직
9,운동화 끈,5000.0,베이직


### 15. ▶️ **코드 실행하기 · 코드 셀 18 [C18]**

In [18]:
sql(r"""
-- [C18]
SELECT
  COUNT(*) AS 주문건수,
  SUM(amount) AS 총매출,
  AVG(amount) AS 평균금액,
  MAX(amount) AS 최고금액,
  MIN(amount) AS 최저금액
FROM orders
WHERE
  amount IS NOT NULL
""")

,주문건수,총매출,평균금액,최고금액,최저금액
0,27,3806000.0,140962.962963,410000.0,32000.0


### 16. ▶️ **코드 실행하기 · 코드 셀 19 [C19]**

In [19]:
sql(r"""
-- [C19]
SELECT
  COUNT(*) AS 전체행수,
  COUNT(amount) AS 금액있는행수
FROM orders
""")

,전체행수,금액있는행수
0,30,27


## 17. ⌨️ 백문이 불여일타 (7)

In [20]:
sql(r"""
-- [C20] ⌨️ 백문이 불여일타 (7) — 전체 고객 수 세기

select count(*) as 고객수
from customers
""")

,고객수
0,15


## 18. ⌨️ 백문이 불여일타 (8)

In [21]:
sql(r"""
-- [C21] ⌨️ 백문이 불여일타 (8) — 최고가·최저가·평균가 한 번에

select 
  min(price), 
  max(price), 
  avg(price)
from products
where price is not null
""")

,min(price),max(price),avg(price)
0,5000.0,175000.0,72200.0


### 19. ▶️ **코드 실행하기 · 코드 셀 22 [C22]**

In [22]:
sql(r"""
-- [C22]
SELECT
  status,
  COUNT(*) AS 건수,
  SUM(amount) AS 매출합
FROM orders
GROUP BY
  status
ORDER BY
  매출합 DESC
""")

,status,건수,매출합
0,Paid,13,1909000.0
1,Shipped,8,1595000.0
2,Returned,3,179000.0
3,Placed,3,123000.0
4,Cancelled,3,NaN


### 20. ▶️ **코드 실행하기 · 코드 셀 23 [C23]**

In [23]:
sql(r"""
-- [C23]
SELECT
  category,
  COUNT(*) AS 상품수,
  AVG(price) AS 평균가격
FROM products
GROUP BY
  category
ORDER BY
  평균가격 DESC
""")

,category,상품수,평균가격
0,Boots,2,160000.0
1,Running,2,104000.0
2,Sneakers,2,57000.0
3,Sandals,2,31500.0
4,Accessory,2,8500.0


### 21. ▶️ **코드 실행하기 · 코드 셀 24 [C24]**

In [24]:
sql(r"""
-- [C24]
SELECT
  status,
  SUM(amount) AS 매출합
FROM orders
WHERE
  amount IS NOT NULL /* (행 단계) NULL 금액 제외 */
GROUP BY
  status
HAVING
  SUM(amount) >= 500000 /* (그룹 단계) 매출 50만 이상만 */
ORDER BY
  매출합 DESC
""")

,status,매출합
0,Paid,1909000.0
1,Shipped,1595000.0


## 22. ⌨️ 백문이 불여일타 (9)

In [25]:
sql(r"""
-- [C25] ⌨️ 백문이 불여일타 (9) — 국가별 고객 수 집계와 정렬

select 
  country as 국가, 
  count(customer_id) as "국가별 고객수"
from customers
where country is not null
group by country
order by "국가별 고객수" desc
""")

,국가,국가별 고객수
0,Korea,8
1,USA,3
2,Japan,3


## 23. ⌨️ 백문이 불여일타 (10)

In [26]:
sql(r"""
-- [C26] ⌨️ 백문이 불여일타 (10) — 고객별 총액 20만원 이상 조회

select 
  customer_id,
  sum(amount) as "고객 총액"
from orders
where amount is not null
group by customer_id
having sum(amount) >= 200000 
order by "고객 총액" desc
""")

,customer_id,고객 총액
0,C013,725000.0
1,C007,678000.0
2,C001,380000.0
3,C012,362000.0
4,C003,349000.0
5,C008,314000.0
6,C002,308000.0


## 24. ⌨️ 백문이 불여일타 (11)

In [27]:
sql(r"""
-- [C27] ⌨️ 백문이 불여일타 (11) — 카테고리별 상품 요약 리포트

select 
  category, 
  avg(price), 
  max(price)
from products 
group by category
having count(product_id) >= 2
order by avg(price) desc
""")

,category,avg(price),max(price)
0,Boots,160000.0,175000.0
1,Running,104000.0,119000.0
2,Sneakers,57000.0,65000.0
3,Sandals,31500.0,38000.0
4,Accessory,8500.0,12000.0


## 25. 📒 연습 문제 — 정렬·가공·집계 (10문항)

In [28]:
sql(r"""
-- [C28] 📒 연습 문제 — 정렬·가공·집계 (1/10)

-- 문제: orders를 amount 오름차순으로 정렬하고, order_id는 '주문번호', amount는 '가격'이라는 별칭으로 조회합니다.

select order_id as "주문번호", amount as "가격"
from orders
order by "가격" desc
""")

,주문번호,가격
0,O0009,410000.0
1,O0027,405000.0
2,O0018,320000.0
3,O0024,268000.0
4,O0004,230000.0
5,O0020,215000.0
6,O0030,187000.0
7,O0016,175000.0
8,O0007,158000.0
9,O0013,142000.0


In [29]:
sql(r"""
-- [C29] 📒 연습 문제 — 정렬·가공·집계 (2/10)

-- 문제: orders의 status에서 중복을 제거하고, '상태'라는 별칭을 붙여 조회합니다.

select distinct status as "상태"
from orders
""")

,상태
0,Paid
1,Shipped
2,Returned
3,Cancelled
4,Placed


In [30]:
sql(r"""
-- [C30] 📒 연습 문제 — 정렬·가공·집계 (3/10)

-- 문제: customers에서 이름이 '김'으로 시작하는 고객을 조회합니다.

SELECT *
FROM customers
WHERE name LIKE '김%'
""")

,customer_id,name,country,signup_date,grade
0,C001,김민준,Korea,2023-01-05,Gold


In [31]:
sql(r"""
-- [C31] 📒 연습 문제 — 정렬·가공·집계 (4/10)

-- 문제: products에서 상품명에 '러너'가 포함된 상품을 조회합니다.

SELECT *
FROM products
WHERE product_name LIKE '%러너%'
""")

,product_id,product_name,category,price
0,P01,에어러너,Running,89000.0
1,P05,트레일 러너,Running,119000.0


In [32]:
sql(r"""
-- [C32] 📒 연습 문제 — 정렬·가공·집계 (5/10)

-- 문제: orders에서 amount 200,000원을 기준으로 '고액'/'일반'을 나눈 열(등급)을 함께 조회합니다. (amount가 NULL이면
--       '미확정')

select 
  order_id,
  customer_id,
  order_date,
  status,
  amount,
  case 
    when amount is null then '미확정' 
    when amount >= 200000 then '고액' 
    else '일반' 
  end as 분류
FROM orders
""")

,order_id,customer_id,order_date,status,amount,분류
0,O0001,C001,2023-09-02,Paid,125000.0,일반
1,O0002,C002,2023-09-05,Shipped,89000.0,일반
2,O0003,C001,2023-09-11,Returned,45000.0,일반
3,O0004,C003,2023-09-15,Paid,230000.0,고액
4,O0005,C004,2023-09-20,Cancelled,NaN,미확정
5,O0006,C005,2023-09-25,Shipped,67000.0,일반
6,O0007,C002,2023-10-01,Paid,158000.0,일반
7,O0008,C006,2023-10-04,Placed,32000.0,일반
8,O0009,C007,2023-10-12,Shipped,410000.0,고액
9,O0010,C008,2023-10-19,Paid,99000.0,일반


In [33]:
sql(r"""
-- [C33] 📒 연습 문제 — 정렬·가공·집계 (6/10)

-- 문제: orders에서 order_date의 '연도'만 뽑아 order_id·order_date와 함께 조회합니다.

SELECT
  order_id,
  order_date,
  CAST(strftime('%Y', order_date) AS INTEGER) AS 주문연도
FROM orders
""")

,order_id,order_date,주문연도
0,O0001,2023-09-02,2023
1,O0002,2023-09-05,2023
2,O0003,2023-09-11,2023
3,O0004,2023-09-15,2023
4,O0005,2023-09-20,2023
5,O0006,2023-09-25,2023
6,O0007,2023-10-01,2023
7,O0008,2023-10-04,2023
8,O0009,2023-10-12,2023
9,O0010,2023-10-19,2023


In [34]:
sql(r"""
-- [C34] 📒 연습 문제 — 정렬·가공·집계 (7/10)

-- 문제: status별로 주문 건수와 amount 합계를 구합니다.

select 
  status,
  count(*),
  sum(amount)
FROM orders
group by status
""")

,status,count(*),sum(amount)
0,Cancelled,3,NaN
1,Paid,13,1909000.0
2,Placed,3,123000.0
3,Returned,3,179000.0
4,Shipped,8,1595000.0


In [35]:
sql(r"""
-- [C35] 📒 연습 문제 — 정렬·가공·집계 (8/10)

-- 문제: status별 amount 합계가 500,000원 이상인 상태만 남기고, 합계가 큰 순으로 정렬합니다.

select 
  status,
  count(*),
  sum(amount)
FROM orders
group by status
having sum(amount) >= 500000
order by sum(amount) desc
""")

,status,count(*),sum(amount)
0,Paid,13,1909000.0
1,Shipped,8,1595000.0


In [36]:
sql(r"""
-- [C36] 📒 연습 문제 — 정렬·가공·집계 (9/10)

-- 문제: orders에서 금액이 가장 큰 주문 3건만 조회합니다. (정렬 + 개수 제한)

select 
  *
FROM orders
order by amount desc nulls last
limit 3
""")

,order_id,customer_id,order_date,status,amount
0,O0009,C007,2023-10-12,Shipped,410000.0
1,O0027,C013,2024-01-22,Shipped,405000.0
2,O0018,C013,2023-11-29,Paid,320000.0


In [37]:
sql(r"""
-- [C37] 📒 연습 문제 — 정렬·가공·집계 (10/10)

-- 문제: orders의 전체 주문 건수와 '금액이 있는' 주문 건수를 한 줄에 함께 구합니다.

select 
  count(*),
  count(amount)
FROM orders
""")

,count(*),count(amount)
0,30,27


## 35. 🧪 종합 실습 — 카테고리·상태별 매출 요약

In [38]:
sql(r"""
-- [C38] 🧪 종합 실습 — 카테고리·상태별 매출 요약 (1/3)

SELECT
  date(order_date, 'start of month') AS 월,
  COUNT(*) AS 주문건수,
  SUM(amount) AS 매출합
FROM orders
WHERE
  amount IS NOT NULL
GROUP BY
  월
ORDER BY
  월 NULLS LAST
""")

,월,주문건수,매출합
0,2023-09-01,5,556000.0
1,2023-10-01,5,775000.0
2,2023-11-01,6,839000.0
3,2023-12-01,5,526000.0
4,2024-01-01,4,850000.0
5,2024-02-01,2,260000.0


## 36. **질문 2.** 위 월별 매출을 그래프로 그려, 추이를 한눈에 봅니다.

In [39]:
sql(r"""
-- [C39] 🧪 종합 실습 — 카테고리·상태별 매출 요약 (2/3)

SELECT
  date(order_date, 'start of month') AS month,
  SUM(amount) AS revenue
FROM orders
WHERE
  amount IS NOT NULL
GROUP BY
  month
ORDER BY
  month NULLS LAST
""")

,month,revenue
0,2023-09-01,556000.0
1,2023-10-01,775000.0
2,2023-11-01,839000.0
3,2023-12-01,526000.0
4,2024-01-01,850000.0
5,2024-02-01,260000.0


## 37. **질문 3.** 고객 등급별로 몇 명인지, 그리고 가장 많은 등급이 무엇인지 확인합니다.

In [40]:
sql(r"""
-- [C40] 🧪 종합 실습 — 카테고리·상태별 매출 요약 (3/3)

SELECT
  grade,
  COUNT(*) AS 고객수
FROM customers
GROUP BY
  grade
ORDER BY
  고객수 DESC
""")

,grade,고객수
0,Silver,5
1,Bronze,5
2,Gold,4
3,NaN,1


## 38. 코드 퀴즈

In [41]:
sql(r"""
-- [C41] 코드 퀴즈 — 상태별 평균 주문 금액

SELECT
  status,
  AVG(amount) AS 평균금액
FROM orders
WHERE
  amount IS NOT NULL
GROUP BY
  status
ORDER BY
  평균금액 DESC
""")

,status,평균금액
0,Shipped,199375.000000
1,Paid,146846.153846
2,Returned,59666.666667
3,Placed,41000.000000
